# W06 — Validation & Model Audit

**Lane:** Structured Content Archetype Clustering

This notebook audits the W05 model for honest validation, leakage, failure cases, and claim strength. It uses decision-support language and avoids causal claims.

## 1. Two Paper Findings + My Methodology Questions

**Source note:** The course research-paper file was not available among the supplied materials. Do not present the following as verified paper findings or quotations. Replace the headings with the exact two findings from the assigned paper before submission.

### Finding 1 — Reported performance result
**Methodology question:** Where does the outcome/label used for the reported result come from, and is its construction strictly based on information available at the intended evaluation time?

**Why it matters:** If label construction uses later information, reported performance may be optimistic.

### Finding 2 — Reported generalization result
**Methodology question:** Does the validation design prevent related observations from the same entity or observation window from appearing across development and evaluation data?

**Why it matters:** Otherwise the evaluation can measure reuse of entity-specific patterns rather than generalization to genuinely unseen entities or periods.

## 2. My W05 Model Under an Honest Split

W05 is unsupervised, so there is no ordinary classification accuracy. The audit uses a **grouped-by-client** split. Entire clients are assigned to development or held-out data. Preprocessing is fitted only on development clients; held-out clients are transformed using those fitted objects. K-Means is fit only on development clients, then held-out content is assigned to the learned centroids.

This gives a before/after comparison between the original W05 full-population reference and an unseen-client evaluation.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

candidates = [
    Path('../outputs/content_archetypes_clustered.parquet'),
    Path('../outputs/content_level_model_dataset.parquet'),
]
data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError('W05 output not found. Run W05 first and keep its parquet output under work/outputs/.')
model_df = pd.read_parquet(data_path)
print('Loaded:', data_path.resolve())
print('Shape:', model_df.shape)

In [ ]:
core_features = [
    'search_volume','word_count','content_age_days','days_since_update',
    'impressions_90d','ctr_90d','avg_position_90d','engagement_rate'
]
required = ['client_hash_id','content_hash_id'] + core_features
missing = [x for x in required if x not in model_df.columns]
if missing:
    raise ValueError(f'Missing required W05 columns: {missing}')
print('W05 feature set verified.')

In [ ]:
W05_K = 3
W05_SILHOUETTE_REFERENCE = 0.8414
print('W05 reference K:', W05_K)
print('W05 reference silhouette:', W05_SILHOUETTE_REFERENCE)

In [ ]:
rng = np.random.RandomState(42)
clients = model_df['client_hash_id'].dropna().unique().copy()
rng.shuffle(clients)
n_dev = int(len(clients) * 0.80)
dev_clients = set(clients[:n_dev])
holdout_clients = set(clients[n_dev:])
dev_mask = model_df['client_hash_id'].isin(dev_clients)
holdout_mask = model_df['client_hash_id'].isin(holdout_clients)
dev_df = model_df.loc[dev_mask].copy()
holdout_df = model_df.loc[holdout_mask].copy()
assert not (set(dev_df.client_hash_id) & set(holdout_df.client_hash_id))
print('Clients:', len(clients))
print('Development clients:', len(dev_clients))
print('Holdout clients:', len(holdout_clients))
print('Development rows:', len(dev_df))
print('Holdout rows:', len(holdout_df))

In [ ]:
def raw_features(df):
    X = df[core_features].copy()
    X['avg_position_90d'] = X['avg_position_90d'].replace(0, np.nan)
    return X

X_dev_raw = raw_features(dev_df)
X_holdout_raw = raw_features(holdout_df)
imputer = SimpleImputer(strategy='median')
X_dev_imp = pd.DataFrame(imputer.fit_transform(X_dev_raw), columns=core_features, index=X_dev_raw.index)
X_holdout_imp = pd.DataFrame(imputer.transform(X_holdout_raw), columns=core_features, index=X_holdout_raw.index)

log_features = ['search_volume','word_count','impressions_90d']
for col in log_features:
    X_dev_imp[col] = np.log1p(X_dev_imp[col].clip(lower=0))
    X_holdout_imp[col] = np.log1p(X_holdout_imp[col].clip(lower=0))

scaler = RobustScaler()
X_dev = scaler.fit_transform(X_dev_imp)
X_holdout = scaler.transform(X_holdout_imp)
print('Preprocessing fitted only on development clients.')

In [ ]:
honest_kmeans = KMeans(n_clusters=W05_K, random_state=42, n_init=10)
dev_labels = honest_kmeans.fit_predict(X_dev)
holdout_labels = honest_kmeans.predict(X_holdout)
dev_silhouette = silhouette_score(X_dev, dev_labels, sample_size=min(50000,len(X_dev)), random_state=42)
holdout_silhouette = silhouette_score(X_holdout, holdout_labels, sample_size=min(50000,len(X_holdout)), random_state=42)
print('Development silhouette:', round(dev_silhouette,4))
print('Held-out silhouette:', round(holdout_silhouette,4))

In [ ]:
dev_sizes = pd.Series(dev_labels).value_counts().sort_index()
holdout_sizes = pd.Series(holdout_labels).value_counts().sort_index()
validation_sizes = pd.DataFrame({
    'development_n': dev_sizes,
    'development_pct': (dev_sizes/len(dev_labels)*100).round(2),
    'holdout_n': holdout_sizes,
    'holdout_pct': (holdout_sizes/len(holdout_labels)*100).round(2),
}).fillna(0)
display(validation_sizes)

In [ ]:
before_after = pd.DataFrame([
    {'evaluation':'W05 full-population reference','silhouette':W05_SILHOUETTE_REFERENCE},
    {'evaluation':'Grouped development clients','silhouette':dev_silhouette},
    {'evaluation':'Grouped held-out clients','silhouette':holdout_silhouette},
])
before_after['difference_vs_W05'] = (before_after['silhouette']-W05_SILHOUETTE_REFERENCE).round(4)
display(before_after)

## 3. Leakage Audit

Core model features: search demand, content structure, freshness, visibility, and engagement. IDs are identifiers only. Query breadth was excluded from the core model after the W05 missingness audit. Future/trend/label-derived fields are excluded.

For the honest split, the imputer and scaler are fit only on development clients and then applied to held-out clients.

In [ ]:
leakage_audit = pd.DataFrame([
 {'feature_or_group':'client_hash_id','model_input':False,'status':'SAFE','reason':'Grouping/joining only'},
 {'feature_or_group':'content_hash_id','model_input':False,'status':'SAFE','reason':'Identification only'},
 {'feature_or_group':'search_volume / word_count','model_input':True,'status':'CHECKED','reason':'Snapshot-available content/search signals'},
 {'feature_or_group':'content_age_days / days_since_update','model_input':True,'status':'CHECKED','reason':'Derived from snapshot-safe dates'},
 {'feature_or_group':'impressions_90d / ctr_90d / avg_position_90d / engagement_rate','model_input':True,'status':'CHECKED','reason':'Historical 90-day observed signals'},
 {'feature_or_group':'query_count_90d','model_input':False,'status':'EXCLUDED','reason':'High missingness could drive data-coverage clusters'},
 {'feature_or_group':'future/trend labels','model_input':False,'status':'EXCLUDED','reason':'Future or label-derived information'},
])
display(leakage_audit)

In [ ]:
print('Development rows used to fit imputer:', len(X_dev_raw))
print('Held-out rows transformed, not used to fit preprocessing:', len(X_holdout_raw))
print('Grouped split leakage: PASS')


In [ ]:
date_results = {}
for col in ['content_created_date','content_updated_date']:
    if col in model_df.columns:
        s = pd.to_datetime(model_df[col], errors='coerce')
        date_results[f'future_{col}'] = int((s > pd.Timestamp('2026-06-30')).sum())
for col in ['content_age_days','days_since_update']:
    if col in model_df.columns:
        date_results[f'negative_{col}'] = int((model_df[col] < 0).sum())
display(pd.Series(date_results, name='count').to_frame())

## 4. Failure / Weak Assignment Review

Weak assignments are content items farthest from their assigned centroid. They are diagnostic examples, not ground-truth classification errors.

In [ ]:
holdout_distances = honest_kmeans.transform(X_holdout)
review = holdout_df[['client_hash_id','content_hash_id'] + core_features].copy()
review['assigned_cluster'] = holdout_labels
review['distance_to_centroid'] = holdout_distances[np.arange(len(holdout_labels)), holdout_labels]
weak_assignments = review.sort_values('distance_to_centroid', ascending=False).head(20)
display(weak_assignments)

In [ ]:
display(weak_assignments['assigned_cluster'].value_counts().sort_index().rename('weak_assignment_count').to_frame())

### Failure interpretation

Review the displayed rows for extreme values, mixed signals, missingness effects, or genuine boundary content. Do not call a weak assignment a model error unless an external reference establishes what the correct cluster should be.

## 5. Claim Rewrite

### Claim 1
**Too strong:** The model identifies content that will increase traffic.

**Rewritten:** The clustering identifies observed groups of content with different search-performance and freshness profiles that can support content-prioritization decisions.

### Claim 2
**Too strong:** Updating stale content will improve performance.

**Rewritten:** One observed group contains older content and weaker performance characteristics, making it a candidate for freshness review; the analysis does not establish a causal effect of updating.

### Claim 3
**Too strong:** The model finds the best content.

**Rewritten:** The model identifies a small group with comparatively stronger observed CTR and search-position characteristics in this dataset snapshot.

### Claim 4
**Too strong:** The model proves which content strategy works.

**Rewritten:** The model provides directional, observed patterns that can support content review and prioritization.

## 6. Limitations and Final Interpretation

- This is unsupervised learning with no ground-truth archetype label.
- Silhouette is an internal separation metric, not business impact or accuracy.
- Median imputation may influence cluster boundaries when missingness is substantial.
- A 90-day snapshot does not represent every content lifecycle.
- Grouped-by-client validation tests generalization to unseen clients, not future time periods.
- A small cluster should be treated as a rare pattern, not a broad portfolio segment.
- The analysis describes observed associations and does not establish causal effects on ranking, traffic, or content changes.

### Public-safe conclusion
The W05 model is best presented as decision-support analysis that identifies recurring content/search-performance patterns. The W06 audit tests whether that structure remains reasonably coherent for unseen clients and documents the assumptions, leakage controls, and ambiguous cases that should accompany downstream recommendations.

## 7. Self-Check

Before submission, confirm:

- [ ] Two exact findings from the assigned paper are inserted into Section 1.
- [ ] Each finding has a constructive methodology question.
- [ ] No client overlap exists between development and held-out groups.
- [ ] Imputer and scaler are fit only on development clients.
- [ ] Held-out clients are assigned using development-fitted centroids.
- [ ] Before/after validation results are shown.
- [ ] Feature leakage audit is complete.
- [ ] Future-information audit is complete.
- [ ] 10–20 weak assignments are shown and reviewed.
- [ ] Claims use observed/directional/decision-support language.
- [ ] Limitations are documented.
- [ ] Notebook runs top-to-bottom without errors.

**Deliverable:** `work/notebooks/w06_validation_audit.ipynb`